# 01 – Collect chatbot answers

Runs all 54 prompts in `dataset/prompts.csv` on two chatbots (`llama3.1:8b`, `qwen3.5:9b`, both local via Ollama), 2 replicates each (54 × 2 × 2 = **216 answers**).

**Setup**
1. Install Ollama, run `ollama pull llama3.1:8b` and `ollama pull qwen3.5:9b`, and keep it running.
2. Gemini (`gemini-3.8-flash`) was dropped: the free tier allows only 20 requests/day. Its 21 partial answers stay in `answers.jsonl` but are not part of the analysis. To use Gemini anyway, add it back to `CHATBOTS` with a GEMINI_API_KEY in `.env`.

**Outputs** (append-only, so the notebook is resumable – rerun it to fill any failed rows)
- `dataset/answers/answers.jsonl` – one line per answer with chatbot, model version, date, settings
- `dataset/answers/run_metadata.json` – one entry per chatbot per run (Ollama server version + model digest, Gemini model resource/version, settings, hash of `prompts.csv`)

In [1]:
import hashlib, json, os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential
from tqdm.auto import tqdm

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")

PROMPTS_CSV = ROOT / "dataset" / "prompts.csv"
OUT_DIR = ROOT / "dataset" / "answers"
ANSWERS_JSONL = OUT_DIR / "answers.jsonl"
RUN_META_JSON = OUT_DIR / "run_metadata.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)

/opt/miniconda3/envs/nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
N_REPLICATES = 2
OLLAMA_URL = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

# Same sampling settings for both chatbots so they are comparable.
# temperature > 0 so the two replicates can differ; seed = replicate number so reruns are reproducible
# (exact for Ollama on the same build; best-effort for Gemini).
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_OUTPUT_TOKENS = 2048   # ~250 words is ~350 tokens; generous headroom

CHATBOTS = {
    "llama3.1:8b":      {"provider": "ollama", "model": "llama3.1:8b"},
    "qwen3.5:9b":       {"provider": "ollama", "model": "qwen3.5:9b"},
    # "gemini-3.8-flash": {"provider": "gemini", "model": "gemini-3.8-flash"},   # dropped: free-tier quota (20 req/day)
}

LIMIT = None   # set to e.g. 2 for a smoke test on the first 2 prompts

def settings_for(rep):
    return {"temperature": TEMPERATURE, "top_p": TOP_P, "max_output_tokens": MAX_OUTPUT_TOKENS,
            "seed": rep, "system_prompt": None}

## Chatbot clients

In [3]:
class OllamaChat:
    def __init__(self, model):
        self.model = model

    def describe(self):
        """Server version + model digest, so the exact weights can be identified later."""
        version = requests.get(f"{OLLAMA_URL}/api/version", timeout=10).json().get("version")
        info = requests.post(f"{OLLAMA_URL}/api/show", json={"model": self.model}, timeout=30)
        info.raise_for_status()
        info = info.json()
        tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json().get("models", [])
        digest = next((m.get("digest") for m in tags if m.get("name") == self.model), None)
        return {"server_version": version, "model_digest": digest,
                "model_details": info.get("details"), "modified_at": info.get("modified_at")}

    @retry(stop=stop_after_attempt(4), wait=wait_exponential(min=2, max=30), reraise=True)
    def generate(self, prompt, s):
        payload = {
            "model": self.model, "stream": False,
            "messages": [{"role": "user", "content": prompt}],
            "options": {"temperature": s["temperature"], "top_p": s["top_p"],
                        "num_predict": s["max_output_tokens"], "seed": s["seed"]},
        }
        # Qwen3.x is a reasoning model: switch thinking off so the answer is plain prose like Llama's
        # and reasoning tokens don't eat the output budget. Llama does not get the parameter.
        thinking_disabled = self.model.startswith("qwen")
        if thinking_disabled:
            payload["think"] = False
        r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=600, json=payload)
        r.raise_for_status()
        j = r.json()
        return {"answer": j["message"]["content"], "model_version_reported": j.get("model"),
                "finish_reason": j.get("done_reason"), "thinking_disabled": thinking_disabled,
                "tokens_in": j.get("prompt_eval_count"), "tokens_out": j.get("eval_count")}


class GeminiChat:
    def __init__(self, model):
        from google import genai
        key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
        if not key:
            raise RuntimeError("GEMINI_API_KEY not found. Put it in a .env file in the repo root (see .env.example).")
        self.client = genai.Client(api_key=key)
        self.model = model

    def describe(self):
        m = self.client.models.get(model=self.model)   # fails fast if the model id is wrong
        return {"resource_name": m.name, "display_name": m.display_name, "version": m.version,
                "input_token_limit": m.input_token_limit, "output_token_limit": m.output_token_limit}

    # Don't retry a daily-quota 429: it won't clear for hours. Other errors (rate spikes, 5xx) still retry.
    @retry(stop=stop_after_attempt(5), wait=wait_exponential(min=4, max=60), reraise=True,
           retry=retry_if_exception(lambda e: "PerDay" not in str(e)))
    def generate(self, prompt, s):
        from google.genai import types
        resp = self.client.models.generate_content(
            model=self.model, contents=prompt,
            config=types.GenerateContentConfig(temperature=s["temperature"], top_p=s["top_p"],
                                               max_output_tokens=s["max_output_tokens"], seed=s["seed"]))
        if not resp.text:
            raise RuntimeError("Empty response, finish_reason="
                               f"{resp.candidates[0].finish_reason if resp.candidates else None}")
        u = resp.usage_metadata
        return {"answer": resp.text, "model_version_reported": resp.model_version,
                "finish_reason": str(resp.candidates[0].finish_reason) if resp.candidates else None,
                "tokens_in": u.prompt_token_count if u else None,
                "tokens_out": u.candidates_token_count if u else None}


def make_client(name):
    cfg = CHATBOTS[name]
    return OllamaChat(cfg["model"]) if cfg["provider"] == "ollama" else GeminiChat(cfg["model"])

## Work out what is left to collect

In [4]:
prompts = pd.read_csv(PROMPTS_CSV)
if LIMIT:
    prompts = prompts.head(LIMIT)

def load_done():
    if not ANSWERS_JSONL.exists():
        return set()
    with ANSWERS_JSONL.open() as f:
        return {(r["prompt_id"], r["chatbot"], r["replicate"])
                for r in map(json.loads, filter(str.strip, f))}

done = load_done()
todo = [(row, name, rep)
        for name in CHATBOTS
        for _, row in prompts.iterrows()
        for rep in range(1, N_REPLICATES + 1)
        if (row.prompt_id, name, rep) not in done]
print(f"{len(prompts)} prompts | {len(todo)} answers to collect | {len(done)} already on disk")

54 prompts | 104 answers to collect | 133 already on disk


## Record model versions and settings, then collect

In [5]:
prompts_hash = hashlib.sha256(PROMPTS_CSV.read_bytes()).hexdigest()[:16]
clients, run_meta = {}, []
for name in sorted({n for _, n, _ in todo}):
    clients[name] = make_client(name)
    entry = {"run_started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
             "chatbot": name, "provider": CHATBOTS[name]["provider"],
             "settings": settings_for(1) | {"seed": f"replicate number (1..{N_REPLICATES})"},
             "prompts_csv_sha256_16": prompts_hash,
             "model_info": clients[name].describe()}
    run_meta.append(entry)
    print(name, "->", entry["model_info"])

if run_meta:
    old = json.loads(RUN_META_JSON.read_text()) if RUN_META_JSON.exists() else []
    RUN_META_JSON.write_text(json.dumps(old + run_meta, indent=2))

qwen3.5:9b -> {'server_version': '0.40.9', 'model_digest': 'd63f830f28d515fefea546f213fca7a75e9dd5176ef51a6deabc952d1c99a911', 'model_details': {'parent_model': '', 'format': 'safetensors', 'family': 'qwen3_5', 'families': None, 'parameter_size': '9.7B', 'quantization_level': 'nvfp4', 'runner': 'mlx'}, 'modified_at': '2026-10-23T16:09:45.278830602+01:00'}


In [6]:
failed, quota_hit = [], set()
with ANSWERS_JSONL.open("a") as out:
    for row, name, rep in tqdm(todo, desc="answers"):
        if name in quota_hit:
            continue
        s = settings_for(rep)
        try:
            res = clients[name].generate(row.full_prompt, s)
        except Exception as e:
            failed.append((row.prompt_id, name, rep, str(e)))
            if "PerDay" in str(e):
                quota_hit.add(name)
                tqdm.write(f"{name}: daily quota exhausted, skipping the rest of its answers. Rerun after it resets.")
                continue
            tqdm.write(f"FAILED {row.prompt_id} {name} rep{rep}: {e}")
            continue
        out.write(json.dumps({
            "prompt_id": row.prompt_id, "question_id": row.question_id,
            "topic_frame": row.topic_frame, "wording": row.wording,
            "chatbot": name, "replicate": rep,
            "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "settings": s, "full_prompt": row.full_prompt,
            "n_words": len(res["answer"].split()), **res,
        }, ensure_ascii=False) + "\n")
        out.flush()   # keep progress if interrupted
print(f"done, {len(failed)} failed (rerun this notebook to retry them)")

answers:   0%|          | 0/104 [00:00<?, ?it/s]

answers:   1%|          | 1/104 [00:19<33:02, 19.25s/it]

answers:   2%|▏         | 2/104 [00:36<30:22, 17.87s/it]

answers:   3%|▎         | 3/104 [00:53<29:27, 17.50s/it]

answers:   4%|▍         | 4/104 [01:09<28:08, 16.89s/it]

...

answers:  98%|█████████▊| 102/104 [1:38:44<38:16, 1148.06s/it]

answers:  99%|█████████▉| 103/104 [3:03:04<38:41, 2321.42s/it]

answers: 100%|██████████| 104/104 [3:48:32<00:00, 2443.61s/it]

answers: 100%|██████████| 104/104 [3:48:32<00:00, 131.85s/it] 

done, 0 failed (rerun this notebook to retry them)


## Sanity checks

In [7]:
answers = pd.read_json(ANSWERS_JSONL, lines=True)
answers = answers[answers["chatbot"].isin(CHATBOTS)]   # ignore dropped chatbots (partial Gemini rows)
print(len(answers), "answers; expected", 54 * len(CHATBOTS) * N_REPLICATES)
display(answers.groupby("chatbot").agg(n=("prompt_id", "size"), words_mean=("n_words", "mean"),
                                       words_min=("n_words", "min"), words_max=("n_words", "max")))
display(answers.groupby("chatbot")["finish_reason"].value_counts())   # watch for truncation (length / MAX_TOKENS)
display(answers.groupby("chatbot")["model_version_reported"].unique())

216 answers; expected 216


,n,words_mean,words_min,words_max
chatbot,,,,
llama3.1:8b,108,223.222222,176,263
qwen3.5:9b,108,294.842593,257,359


chatbot      finish_reason
llama3.1:8b  stop             108
qwen3.5:9b   stop             108
Name: count, dtype: int64

chatbot
llama3.1:8b    [llama3.1:8b]
qwen3.5:9b      [qwen3.5:9b]
Name: model_version_reported, dtype: object